# 09 — TomTom Empirical-Resampling Robustness (ERR)

**Exact terminology (never altered)**: *"empirical-resampling robustness
using observed TomTom historical traffic ratios."* This is NOT called
"live traffic replay," "fleet GPS validation," or "real vehicle
trajectory validation" anywhere in this notebook.

Uses only the fully aggregated `historic_ratio` value pool -- no
origin-destination-pair identifiers, no timestamps, no names, no
coordinates (data-minimized; see REPRODUCIBILITY_NOTE.md).

**Scope is NOT equal between methods**: SA/NR are evaluated on the
**full 51 held-out instances**; Liu-ALNS is evaluated on a **stratified
11-instance computational-robustness subset** (every 5th instance from
the sorted 51), disclosed explicitly as such and never described as a
full validation.


In [ ]:
import os, sys, json
import pandas as pd
import numpy as np
assert 'REPO_ROOT' in dir(), "Run notebook 00 first."
sys.path.insert(0, REPO_ROOT)


## Empirical-ratio construction (from de-identified raw TomTom observations)

In [ ]:
DEID_DIR = os.path.join(REPO_ROOT, "data_deidentified")
PRIVATE_DIR = os.path.join(REPO_ROOT, "data_private")
aggregated_path = os.path.join(PRIVATE_DIR, "tomtom_ratio_pool_aggregated_PRIVATE.csv")

# Redistribution permission for TomTom-derived data (even the aggregated
# ratio pool) has not been confirmed -- it lives under data_private/, not
# in the public data_deidentified/ package. See REPRODUCIBILITY_NOTE.md.
if os.path.exists(aggregated_path):
    tomtom = pd.read_csv(aggregated_path)
    print(f"Ratio pool available (private/authorized environment): {len(tomtom)} values")
    ratios = tomtom['historic_ratio'].dropna()
    pooled_median = ratios.median()
    normalized_pool = (ratios / pooled_median).values
    print(f"Pooled median: {pooled_median:.4f}")
    print(f"Normalized pool: {len(normalized_pool)} values, mean={normalized_pool.mean():.4f}")
else:
    normalized_pool = None
    print("TomTom ratio pool not available in this environment (expected in PUBLIC mode --")
    print("redistribution permission not yet confirmed). The pre-computed ERR EXPERIMENT")
    print("RESULTS below (err_nr_sa_raw.csv, err_liu_alns_raw.csv) remain public and")
    print("reproducible, since they are aggregated study outcomes, not the TomTom source")
    print("data itself.")


## Resampling metadata (frozen, disclosed exactly)

In [ ]:
RESAMPLE_METADATA = {
    "sa_nr_seed": 555, "liu_seed": 777, "search_seed": 101,
    "n_resample_sa_nr": 20, "n_resample_liu": 10,
    "sa_nr_scope": "full 51 held-out instances x 3 triggers, seed=101",
    "liu_scope": "STRATIFIED 11-instance computational-robustness subset (NOT full validation)",
    "stratification_rule": "every 5th instance from the sorted 51-instance list",
    "sa_nr_denominator": 3060, "liu_denominator": 330,
    "no_harm_definition": "method_lateness > baseline_lateness + 1e-6 counts as adverse",
}
print(json.dumps(RESAMPLE_METADATA, indent=2))


## Load pre-computed ERR raw outputs (public statistical reproduction)

In [ ]:
EXP_DIR = os.path.join(DEID_DIR, "experiment_outputs")
err_nr_sa = pd.read_csv(os.path.join(EXP_DIR, "err_nr_sa_raw.csv"))
err_liu = pd.read_csv(os.path.join(EXP_DIR, "err_liu_alns_raw.csv"))

print(f"NR/SA ERR: {len(err_nr_sa)} rows, {err_nr_sa['delivery_date'].nunique()} instances, "
      f"{err_nr_sa['rep'].nunique()} resamples")
print(f"Liu-ALNS ERR: {len(err_liu)} rows, {err_liu['delivery_date'].nunique()} instances "
      f"(stratified subset), {err_liu['rep'].nunique()} resamples")

if RUN_MODE == "full":
    assert err_nr_sa['delivery_date'].nunique() == 51, "SA/NR ERR must cover all 51 held-out instances"
    assert err_liu['delivery_date'].nunique() == 11, "Liu-ALNS ERR must use exactly the 11-instance stratified subset"


## Summary statistics (regenerated, not hard-coded)

In [ ]:
nr_sa_block = err_nr_sa.groupby(['delivery_date','trigger_fraction','rep']).agg(
    nr_lateness=('nr_lateness','sum'), sa_lateness=('sa_lateness','sum')).reset_index()
sa_reduction_pct = 100*(1 - nr_sa_block['sa_lateness'].mean()/nr_sa_block['nr_lateness'].mean())
sa_harm = (nr_sa_block['sa_lateness'] > nr_sa_block['nr_lateness']+1e-6).sum()
print(f"SA (n={len(nr_sa_block)} resampled scenarios): mean NR={nr_sa_block['nr_lateness'].mean():.4f}, "
      f"mean SA={nr_sa_block['sa_lateness'].mean():.4f}, reduction={sa_reduction_pct:.2f}%")
print(f"SA empirical no-harm: {100*(1-sa_harm/len(nr_sa_block)):.4f}% ({sa_harm}/{len(nr_sa_block)} adverse)")

liu_block = err_liu.groupby(['delivery_date','trigger_fraction','rep']).agg(
    nr_lateness=('nr_lateness','sum'), liu_lateness=('liu_lateness','sum')).reset_index()
liu_reduction_pct = 100*(1 - liu_block['liu_lateness'].mean()/liu_block['nr_lateness'].mean())
liu_harm = (liu_block['liu_lateness'] > liu_block['nr_lateness']+1e-6).sum()
print(f"\nLiu-ALNS (n={len(liu_block)} resampled scenarios, STRATIFIED SUBSET): "
      f"mean NR={liu_block['nr_lateness'].mean():.4f}, mean Liu={liu_block['liu_lateness'].mean():.4f}, "
      f"reduction={liu_reduction_pct:.2f}%")
print(f"Liu-ALNS empirical no-harm: {100*(1-liu_harm/len(liu_block)):.4f}% ({liu_harm}/{len(liu_block)} adverse)")

results_dir = os.path.join(REPO_ROOT, "results")
os.makedirs(results_dir, exist_ok=True)
err_summary = dict(sa_n=len(nr_sa_block), sa_reduction_pct=sa_reduction_pct, sa_no_harm_pct=100*(1-sa_harm/len(nr_sa_block)),
                    liu_n=len(liu_block), liu_reduction_pct=liu_reduction_pct, liu_no_harm_pct=100*(1-liu_harm/len(liu_block)),
                    metadata=RESAMPLE_METADATA)
with open(os.path.join(results_dir, "err_tomtom_robustness_summary.json"), "w") as f:
    json.dump(err_summary, f, indent=2)
print("\nSaved results/err_tomtom_robustness_summary.json")


## Expected outputs / integrity checks

In [ ]:
checks = {
    "tomtom_pool_check_consistent": (normalized_pool is None) or (len(normalized_pool) == 960),
    "sa_nr_data_loaded": len(err_nr_sa) > 0,
    "liu_data_loaded": len(err_liu) > 0,
    "liu_scope_disclosed_as_subset": err_liu['delivery_date'].nunique() < err_nr_sa['delivery_date'].nunique(),
}
if RUN_MODE == "full":
    checks["sa_full_51_scope"] = err_nr_sa['delivery_date'].nunique() == 51
    checks["liu_exact_11_subset"] = err_liu['delivery_date'].nunique() == 11

for k, v in checks.items():
    print(f"{'PASS' if v else 'FAIL'}  {k}")
NOTEBOOK_09_STATUS = "PASS" if all(checks.values()) else "FAIL"
print(f"\nNOTEBOOK 09 STATUS: {NOTEBOOK_09_STATUS}")
assert NOTEBOOK_09_STATUS == "PASS"
